In [5]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.metrics import make_scorer, accuracy_score
import numpy as np
import xgboost as xgb
import json
import pandas as pd
from pandas import json_normalize
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from textblob import TextBlob
import re
from sklearn.ensemble import RandomForestClassifier
from gensim.models import Word2Vec
from nltk.corpus import stopwords


In [6]:
# ===============================
# JSONL LOADING
# ===============================

train_data = pd.read_json('train.jsonl', lines=True)
train_data = json_normalize(train_data.to_dict(orient='records'))

kaggle_data = pd.read_json('kaggle_test.jsonl', lines=True)
kaggle_data = json_normalize(kaggle_data.to_dict(orient='records'))

X_train = train_data.drop('label', axis=1)
y_train = train_data['label']

X_kaggle = kaggle_data

print("Chargement OK")

Chargement OK


In [7]:
def create_advanced_features(df_input):
    df = df_input.copy()
    
    default_int_series = pd.Series(0, index=df.index)
    default_bool_series = pd.Series(False, index=df.index)
    
    df['user.followers_count'] = df.get('user.followers_count', default_int_series).fillna(0)
    df['user.friends_count'] = df.get('user.friends_count', default_int_series).fillna(0)
    df['user.listed_count'] = df.get('user.listed_count', default_int_series).fillna(0)
    df['user.favourites_count'] = df.get('user.favourites_count', default_int_series).fillna(0)
    df['user.statuses_count'] = df.get('user.statuses_count', default_int_series).fillna(0)
    df['retweet_count'] = df.get('retweet_count', default_int_series).fillna(0)
    df['favorite_count'] = df.get('favorite_count', default_int_series).fillna(0)
    df['quote_count'] = df.get('quote_count', default_int_series).fillna(0) # New : Quote Count
    df['reply_count'] = df.get('reply_count', default_int_series).fillna(0) # New : Reply Count
    
    # Get the dates
    df['user_created_at_dt'] = pd.to_datetime(df.get('user.created_at'), errors='coerce')
    ref_date = pd.to_datetime('now', utc=True)
    df['account_age_days'] = (ref_date - df['user_created_at_dt']).dt.days
    df['account_age_days'] = df['account_age_days'].fillna(0)
    
    # Extraction temporal indic
    df['created_at_dt'] = pd.to_datetime(df.get('created_at'), errors='coerce')
    df['tweet_hour'] = df['created_at_dt'].dt.hour.fillna(-1)
    df['tweet_is_weekend'] = df['created_at_dt'].dt.dayofweek.isin([5, 6]).fillna(False).astype(int)

    # Profile
    df['is_default_profile'] = df.get('user.default_profile', default_bool_series).fillna(False).astype(int)
    df['is_default_image'] = df.get('user.default_profile_image', default_bool_series).fillna(False).astype(int)
    df['is_verified'] = df.get('user.verified', default_bool_series).fillna(False).astype(int)
    df['is_protected'] = df.get('user.protected', default_bool_series).fillna(False).astype(int)
    df['has_url'] = df.get('user.url', pd.Series(False, index=df.index)).notna().astype(int)

    # Content
    def count_entities(x):
        if isinstance(x, list) or (isinstance(x, pd.Series) and x.dtype == object): return len(x)
        return 0

    df['num_urls'] = df.get('entities.urls', default_int_series).apply(count_entities)
    df['num_hashtags'] = df.get('entities.hashtags', default_int_series).apply(count_entities)
    df['num_mentions'] = df.get('entities.user_mentions', default_int_series).apply(count_entities)
    df['has_media'] = df.get('extended_entities.media', default_bool_series).notna().astype(int)

    # RATIOS
    followers = df['user.followers_count']
    friends = df['user.friends_count']
    listed = df['user.listed_count']
    statuses = df['user.statuses_count']
    df['ratio_followers_friends'] = followers / (friends + 1)
    df['ratio_listed_followers'] = listed / (followers + 1)
    df['reciprocity_score'] = (friends - followers) / (friends + followers + 1)
    df['tweets_per_day'] = statuses / (df['account_age_days'] + 1)
    df['ratio_mention_status'] = df['num_mentions'] / (statuses + 1)
    total_engagement = df['retweet_count'] + df['favorite_count'] + df['quote_count'] + df['reply_count']
    df['total_tweet_engagement'] = total_engagement / (followers + 1)

    # Lenght
    df['final_text'] = df.get('extended_tweet.full_text', df.get('text', pd.Series(''))).fillna('')
    df['final_text'] = df['final_text'].where(df['final_text'] != '', df.get('text', '')).fillna('')
    df['text_length'] = df['final_text'].astype(str).apply(len)
    df['bio_length'] = df.get('user.description', '').astype(str).apply(len)

    features_to_keep = [
        'user.followers_count', 'user.friends_count', 'user.listed_count', 
        'user.favourites_count', 'user.statuses_count',
        'retweet_count', 'favorite_count', 'quote_count', 'reply_count',
        'ratio_followers_friends', 'ratio_listed_followers', 'tweets_per_day', 'account_age_days',
        'reciprocity_score', 'ratio_mention_status', 'total_tweet_engagement',
        'is_verified', 'is_default_profile', 'is_default_image', 'is_geo_enabled',
        'is_protected', 'has_url',
        'tweet_hour', 'tweet_is_weekend',
        'text_length', 'bio_length', 
        'num_urls', 'num_hashtags', 'num_mentions', 'has_media',
    ]
    
    final_cols = [c for c in features_to_keep if c in df.columns]
    
    return df[final_cols].fillna(0)

In [8]:
def create_nlp_features(df_train, df_test, y_train):
    # ----------------------------------------
    # Prep of the texts
    # ----------------------------------------

    french_stopwords = stopwords.words("french")

    train_bio = df_train.get('user.description', pd.Series([''] * len(df_train))).fillna('').astype(str)
    test_bio = df_test.get('user.description', pd.Series([''] * len(df_test))).fillna('').astype(str)

    def get_final_text(df):
        text = df.get('text', pd.Series([''] * len(df))).fillna('')
        full_text = df.get('extended_tweet.full_text', text).fillna(text)
        return full_text.astype(str)
        
    train_text = get_final_text(df_train)
    test_text = get_final_text(df_test)

    def clean_text(text):
        text = text.lower()
        text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
        return text

    # ----------------------------------------
    # TF-IDF + log_reg on tweet
    # ----------------------------------------
    train_text_clean = train_text.apply(clean_text)
    test_text_clean = test_text.apply(clean_text)
    tfidf = TfidfVectorizer(max_features=1000, stop_words=french_stopwords, ngram_range=(2, 5), analyzer='char_wb', lowercase=False)
    X_train_tfidf_tweet = tfidf.fit_transform(train_text_clean)
    X_test_tfidf_tweet = tfidf.transform(test_text_clean)
    log_reg = LogisticRegression(solver='sag', random_state=42)
    log_reg.fit(X_train_tfidf_tweet, y_train.astype(int))
    train_tweet_proba = log_reg.predict_proba(X_train_tfidf_tweet)[:, 1]
    test_tweet_proba = log_reg.predict_proba(X_test_tfidf_tweet)[:, 1]    

    # ----------------------------------------
    # TF-IDF + log_reg on bio
    # ----------------------------------------
    train_bio_clean = train_bio.apply(clean_text)
    test_bio_clean = test_bio.apply(clean_text)
    tfidf = TfidfVectorizer(max_features=1000, stop_words=french_stopwords, ngram_range=(2, 5), analyzer='char_wb', lowercase=False)
    X_train_tfidf = tfidf.fit_transform(train_bio_clean)
    X_test_tfidf = tfidf.transform(test_bio_clean)
    log_reg = LogisticRegression(solver='liblinear', random_state=42)
    log_reg.fit(X_train_tfidf, y_train.astype(int))
    train_bio_proba = log_reg.predict_proba(X_train_tfidf)[:, 1]
    test_bio_proba = log_reg.predict_proba(X_test_tfidf)[:, 1]

    # ----------------------------------------
    # Sentiment analysis
    # ----------------------------------------
    def get_sentiment(text):
        try:
            analysis = TextBlob(text)
            return pd.Series({'polarity': analysis.sentiment.polarity, 'subjectivity': analysis.sentiment.subjectivity})
        except:
            return pd.Series({'polarity': 0.0, 'subjectivity': 0.0})
    train_sentiment = train_text.apply(get_sentiment)
    test_sentiment = test_text.apply(get_sentiment)

    # ----------------------------------------
    # Fusion of the features
    # ----------------------------------------
    df_train_nlp = pd.DataFrame({
        'meta_bio_proba': train_bio_proba,
        'tweet_polarity': train_sentiment['polarity'],
        'tweet_subjectivity': train_sentiment['subjectivity'],
        'meta_tweet_proba': train_tweet_proba
    })
    df_test_nlp = pd.DataFrame({
        'meta_bio_proba': test_bio_proba,
        'tweet_polarity': test_sentiment['polarity'],
        'tweet_subjectivity': test_sentiment['subjectivity'],
        'meta_tweet_proba': test_tweet_proba
    })

    return df_train_nlp, df_test_nlp

In [9]:
X_train_advanced = create_advanced_features(X_train)
X_kaggle_advanced = create_advanced_features(X_kaggle)

y_train_clean = y_train.astype(int)
X_train_nlp, X_kaggle_nlp = create_nlp_features(X_train, X_kaggle, y_train_clean)
X_train_advanced = pd.concat([X_train_advanced, X_train_nlp], axis=1)
X_kaggle_advanced = pd.concat([X_kaggle_advanced, X_kaggle_nlp], axis=1)

print(f"\nFeatures combinées ({len(X_train_advanced.columns)}):")
print(list(X_train_advanced.columns))

C:\Users\hp1ma\AppData\Local\Temp\ipykernel_14780\2501392516.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['user_created_at_dt'] = pd.to_datetime(df.get('user.created_at'), errors='coerce')
C:\Users\hp1ma\AppData\Local\Temp\ipykernel_14780\2501392516.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['user_created_at_dt'] = pd.to_datetime(df.get('user.created_at'), errors='coerce')
c:\Users\hp1ma\anaconda3\envs\inf554\Lib\site-packages\sklearn\feature_extraction\text.py:543: UserWarning: The parameter 'stop_words' will not be used since 'analyzer' != 'word'
  warnings.warn(
c:\Users\hp1ma\anaconda3\envs\inf554\Lib\site-packages\sklearn\feature_extraction\text.py:543: UserWarning: The parameter 's


Features combinées (33):
['user.followers_count', 'user.friends_count', 'user.listed_count', 'user.favourites_count', 'user.statuses_count', 'retweet_count', 'favorite_count', 'quote_count', 'reply_count', 'ratio_followers_friends', 'ratio_listed_followers', 'tweets_per_day', 'account_age_days', 'reciprocity_score', 'ratio_mention_status', 'total_tweet_engagement', 'is_verified', 'is_default_profile', 'is_default_image', 'is_protected', 'has_url', 'tweet_hour', 'tweet_is_weekend', 'text_length', 'bio_length', 'num_urls', 'num_hashtags', 'num_mentions', 'has_media', 'meta_bio_proba', 'tweet_polarity', 'tweet_subjectivity', 'meta_tweet_proba']


On ajoute les embedding par Word2vec

In [10]:
def simple_tokenize(text):
    text = text.lower()
    text = re.sub(r'[.,;:`"\'!?()]', ' ', text)
    return [word for word in text.split() if word]

def extract_full_text(row):
    """Extrait le texte le plus complet disponible du tweet."""
    if "extended_tweet.full_text" in row and pd.notna(row["extended_tweet.full_text"]):
        return row["extended_tweet.full_text"]
    if "text" in row and pd.notna(row["text"]):
        return row["text"]
    return ""

try:
    train_data["full_text"] = train_data.apply(extract_full_text, axis=1)
    kaggle_data["full_text"] = kaggle_data.apply(extract_full_text, axis=1)
    X_full = train_data["full_text"].values
    y_full = train_data["label"].values.astype(int) 
    X_kaggle_full = kaggle_data["full_text"].values
    
    print("✓ Données brutes (X_full, y_full) chargées et prêtes.")

except FileNotFoundError as e:
    print(f"ERREUR: Fichier introuvable. Vérifiez votre chemin : {e}")
    exit()

✓ Données brutes (X_full, y_full) chargées et prêtes.


In [11]:
EMBEDDING_DIM = 250 
WINDOW_SIZE = 50
MIN_COUNT = 1


X_full_tokenized = [simple_tokenize(text) for text in X_full]
X_kaggle_tokenized = [simple_tokenize(text) for text in X_kaggle_full]

def document_vectorizer(tokens, model, dim):
    vector = np.zeros(dim)
    count = 0
    for word in tokens:
        if word in model.wv:
            vector += model.wv[word]
            count += 1
    if count != 0:
        vector /= count
        
    return vector

w2v_model = Word2Vec(
    sentences=X_full_tokenized, 
    vector_size=EMBEDDING_DIM, 
    window=WINDOW_SIZE, 
    min_count=MIN_COUNT, 
    sg=1)

X_train_vectors = np.array([document_vectorizer(tokens, w2v_model, EMBEDDING_DIM) for tokens in X_full_tokenized])
X_kaggle_vectors = np.array([document_vectorizer(tokens, w2v_model, EMBEDDING_DIM) for tokens in X_kaggle_tokenized])

embed_cols = [f'w2v_e_{i}' for i in range(EMBEDDING_DIM)]

X_train_nlp = pd.DataFrame(X_train_vectors, columns=embed_cols)
X_kaggle_nlp = pd.DataFrame(X_kaggle_vectors, columns=embed_cols)
X_train_advanced = pd.concat([X_train_advanced, X_train_nlp], axis=1)
X_kaggle_advanced = pd.concat([X_kaggle_advanced, X_kaggle_nlp], axis=1)

print(f"Total Features d'entraînement (X_train_advanced) : {len(X_train_advanced.columns)}")
print(f"Total Features de test (X_kaggle_advanced) : {len(X_kaggle_advanced.columns)}")

Total Features d'entraînement (X_train_advanced) : 283
Total Features de test (X_kaggle_advanced) : 283


Xgboost final

In [12]:
optimal_params = {
    'colsample_bytree': 0.7069187275124247,
    'reg_alpha': 0.5247746602583891, 
    'reg_lambda': 1.9993048585762774, 
    'learning_rate': 0.01653319284990616,
    'min_child_weight': 79, 
    'n_estimators': 489,
    'max_depth': 91, 
    'subsample': 0.935552788417904,
}

# Best parameters
xgb_model = xgb.XGBClassifier(
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    tree_method='hist',
    colsample_bytree=optimal_params['colsample_bytree'],
    reg_alpha=optimal_params['reg_alpha'],
    reg_lambda=optimal_params['reg_lambda'],
    learning_rate=optimal_params['learning_rate'],
    min_child_weight=optimal_params['min_child_weight'],
    n_estimators=optimal_params['n_estimators'],
    max_depth=optimal_params['max_depth'],
    subsample=optimal_params['subsample'],
)

xgb_model.fit(X_train_advanced, y_train_clean)
y_pred_kaggle = xgb_model.predict(X_kaggle_advanced)

# =======================================================
# GÉNÉRATION submission .csv
# =======================================================
output = pd.concat([X_kaggle['challenge_id'], pd.DataFrame(y_pred_kaggle)], axis=1, ignore_index=True)
output.columns = ['ID', "Prediction"]
output.to_csv('submission_xgboost_without_vote.csv', index=False)

print("\n Fichier 'submission_xgboost_without_vote.csv' generated !")
print(output.head())

c:\Users\hp1ma\anaconda3\envs\inf554\Lib\site-packages\xgboost\training.py:199: UserWarning: [15:40:18] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



 Fichier 'submission_xgboost_without_vote.csv' generated !
   ID  Prediction
0   0           1
1   2           1
2   4           0
3   8           1
4   9           0


With the previous fil we get on kaggle 0.840 and by adding the following majority vote code it reachs to 0.842.

In [13]:
USER_ID_COLUMN = 'user.profile_banner_url'

df_kaggle_preds = pd.DataFrame({
    'challenge_id': X_kaggle['challenge_id'],
    'user_id_key': X_kaggle[USER_ID_COLUMN],
    'y_pred_tweet': y_pred_kaggle           # Prédiction individuelle (le secours)
})

user_pred_mean = df_kaggle_preds.groupby('user_id_key')['y_pred_tweet'].mean()
user_majority_vote = np.where(user_pred_mean >= 0.5, 1, 0)

df_majority_vote = pd.DataFrame({
    'user_id_key': user_pred_mean.index,
    'y_pred_user_majority': user_majority_vote
})

df_final_preds = pd.merge(
    df_kaggle_preds,
    df_majority_vote,
    on='user_id_key',
    how='left'
)

df_final_preds['y_pred_final'] = df_final_preds['y_pred_user_majority'].fillna(
    df_final_preds['y_pred_tweet']
)

y_final_submission = df_final_preds['y_pred_final']
output = pd.DataFrame({
    'ID': df_final_preds['challenge_id'],
    "Prediction": y_final_submission
})

output['Prediction'] = output['Prediction'].astype(int)
output.to_csv('submission_xgboost_with_majority_vote_final.csv', index=False)

print("\n Fichier 'submission_xgboost_with_majority_vote_final.csv' generated !")
print(output.head())


 Fichier 'submission_xgboost_with_majority_vote_final.csv' generated !
   ID  Prediction
0   0           1
1   2           1
2   4           0
3   8           1
4   9           0
